In [1]:
from wrappers import *
from config import *

In [2]:
args = {'geometry':'sphere', \
    'percentage': 2,\
    'strategy': 'random',\
    'niterations': 100, \
    'maptag': '50'}
polydata = fetch_polydata(geometry='sphere', space = 'fsLR-4k', datasource = 'grf')
desc,all_sampling = fetch_sample(**args)

In [11]:
import joblib
datasource = 'grf'

for percentage in test_percentages:
    for maptag in test_ranges:
        args = {'geometry':'sphere', \
        'percentage': percentage,\
        'strategy': 'random',\
        'niterations': 100, \
        'maptag': str(maptag)}
        
        joblib.dump( 
            (steps , recombine_iterations(datasource='grf', niters=args['niterations'], \
                              maptag='50', \
                              outdir=outdir,\
                              tags=args,\
                              verbose=1) ), 
            outdir + f"/{template}_{args['maptag']}_{datasource}_samp-{args['strategy']}_pct-{args['percentage']}.pkl"
        )

In [12]:
import joblib

joblib.load(outdir + 'fsLR-4k_100_grf_samp-random_pct-10.pkl')

(['voronoi', 'variogram', 'determ_params', 'interpolated', 'performance'],
 ({0: (array([198.8321363 , 451.55463266,  97.70898426, 200.10382344,
           278.37764548, 243.36565201, 189.98226164, 254.42544487,
           148.53781158, 330.98841966, 184.28517066, 274.36260351,
           521.35459595, 313.17870021, 146.65512023, 131.54824587,
           172.48095041, 189.54310033, 178.09110542, 296.71768571,
           148.21785127, 964.84646214, 357.6341194 , 631.72087753,
           197.35419969, 655.8485835 , 130.35073579, 223.40342115,
           198.70573254, 304.48720321, 177.98663573, 239.24341028,
           570.07194028, 666.66423225, 335.15236934, 383.95711397,
           246.3973561 , 356.13383969, 133.59221435, 370.35982159,
           230.76040155, 352.98571626, 230.76127105, 176.78249286,
           217.94638027, 310.84501211, 230.38951452, 183.12648868,
           188.64965842, 663.09144459, 244.39423884, 477.82412433,
           577.07113654, 577.61982628, 382.97066132

In [3]:

print(desc)
print(polydata.array_names)


('known_coords', 'known_verts', 'unknown_coords', 'unknown_verts')
['field', '25', '50', '75', '100']


In [4]:
Yk0 = generate_covariates(polydata[args['maptag']], polydata, 100, 'grf')


/home/yzhou/anaconda3/envs/py311/lib/python3.11/site-packages/neuromaps/datasets/utils.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


0.2748338870523338


In [5]:
import sys
sys.path.append('../')
from interpmodules import deterministic

In [5]:
ground_truth = polydata[args['maptag']]
curr_iter=0
# unpack variables
all_known_coords, all_known_verts, all_unknown_coords, all_unknown_verts = all_sampling
known_coords = all_known_coords[:,:,curr_iter]
known_verts = all_known_verts[:,curr_iter].astype(int)
unknown_coords = all_unknown_coords[:,:,curr_iter]
unknown_verts = all_unknown_verts[:,curr_iter].astype(int)

In [7]:
input_vals = ground_truth[known_verts].flatten()
input_coords = known_coords

X0 = input_coords
x1 = unknown_coords
Y0 = input_vals
y0 = ground_truth[unknown_verts].flatten() # actual values



tmp = deterministic.interpolate_RBF(X0, Y0, x1, n_neighbours=10, kernel='cubic')


In [7]:
import copy
from interpmodules import geospatial


for curr_iter in range(2):

    unknown_verts = all_unknown_verts[:,curr_iter].astype(int)

    X0_extended = polydata.points
    Y0_extended = copy.deepcopy(ground_truth)
    Y0_extended[unknown_verts] = 0
    Yk_extended = Yk0



    geospatial.smoothing_over_GWR(X0=X0_extended, Y0=Y0_extended, Yk0=Yk_extended, bandwidth=None, timeit=True)


Note: not learning new coefficient, using coefficients at KNOW LOCATIONS to smooth over ZEROED UNKNOWN VALUES
Note: not learning new coefficient, using coefficients at KNOW LOCATIONS to smooth over ZEROED UNKNOWN VALUES


In [ ]:
from joblib import Parallel, delayed


Parallel(n_jobs=100, backend='loky')(
    delayed(deterministic.interpolate_and_optimize)(
        method='rbf',
        X0=all_known_coords[:, :, curr_iter],
        Y0=ground_truth[all_known_verts[:, curr_iter].astype(int)],
        x1=all_unknown_coords[:, :, curr_iter],
        ground_truth=ground_truth[all_unknown_verts[:, curr_iter].astype(int)].flatten(),
        bounds=[2, 20]
    )
    for curr_iter in range(100)
)

In [22]:
rslt

Parallel(n_jobs=100)